# 🏆 Capstone 1: RAG Knowledge Assistant

---

## 📖 The Brief

Build a **full-stack RAG system** that ingests documents, answers questions with citations, handles follow-up questions, and evaluates its own answer quality.

**Skills tested**: Chapters 3-5 (LangChain, Embeddings, RAG) + Chapter 10 (Production)

---

## 🎯 Requirements

```
╔══════════════════════════════════════════════════════════════════════╗
║  ✅ Ingest documents (PDF, web pages, text)                       ║
║  ✅ Chunk and embed with overlap                                   ║
║  ✅ Retrieve relevant chunks (hybrid: BM25 + vector)              ║
║  ✅ Generate answers with source citations                         ║
║  ✅ Handle follow-up questions (conversation memory)               ║
║  ✅ Evaluate answer quality (faithfulness + relevance)             ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# !pip install langchain-openai langchain-community chromadb python-dotenv rank_bm25 beautifulsoup4 pydantic

## 🧪 Part 1: Document Ingestion Pipeline

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

load_dotenv()

# Config
CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
COLLECTION = "capstone_rag"

# Load sample documents (web pages)
urls = [
    "https://python.langchain.com/docs/concepts/rag/",  
]

# For demo, create synthetic documents
sample_docs = [
    Document(page_content="RAG (Retrieval Augmented Generation) combines retrieval with generation. "
             "The pipeline has three stages: indexing, retrieval, and generation. "
             "Indexing involves loading documents, splitting them into chunks, and storing embeddings in a vector database.",
             metadata={"source": "rag_guide.md", "section": "overview"}),
    Document(page_content="Vector databases store embeddings for fast similarity search. "
             "Popular options include ChromaDB (prototyping), FAISS (high performance), "
             "and Pinecone (managed cloud). HNSW algorithm enables O(log n) search.",
             metadata={"source": "vector_db.md", "section": "databases"}),
    Document(page_content="Advanced RAG techniques include multi-query retrieval, HyDE, "
             "hybrid search combining BM25 with vector search, and re-ranking with cross-encoders. "
             "These techniques improve retrieval precision significantly.",
             metadata={"source": "advanced_rag.md", "section": "techniques"}),
    Document(page_content="LangChain Expression Language (LCEL) uses pipe syntax to compose chains. "
             "Example: prompt | llm | parser. LCEL provides automatic streaming, async, "
             "and batch support. Key runnables: RunnablePassthrough, RunnableLambda, RunnableParallel.",
             metadata={"source": "langchain.md", "section": "lcel"}),
    Document(page_content="LangGraph extends LangChain with stateful graph workflows. "
             "Core concepts: State (TypedDict), Nodes (functions), Edges (connections). "
             "Supports cycles for agent loops, checkpointing for persistence, "
             "and human-in-the-loop via interrupt_before/interrupt_after.",
             metadata={"source": "langgraph.md", "section": "fundamentals"}),
]

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(sample_docs)
print(f"📄 {len(sample_docs)} documents → {len(chunks)} chunks")

# Store in ChromaDB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION
)
print(f"✅ Stored in ChromaDB collection '{COLLECTION}'")

## 🧪 Part 2: Hybrid Retriever (BM25 + Vector)

In [ ]:
from rank_bm25 import BM25Okapi
from typing import List

class HybridRetriever:
    """Combines BM25 keyword search with vector semantic search."""
    
    def __init__(self, vectorstore, documents: List[Document], k: int = 3):
        self.vectorstore = vectorstore
        self.documents = documents
        self.k = k
        
        # Build BM25 index
        tokenized = [doc.page_content.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
    
    def retrieve(self, query: str) -> List[Document]:
        # Vector search
        vector_results = self.vectorstore.similarity_search(query, k=self.k)
        
        # BM25 search
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:self.k]
        bm25_results = [self.documents[i] for i in top_bm25_idx]
        
        # RRF Fusion
        return self._rrf_fusion(vector_results, bm25_results)
    
    def _rrf_fusion(self, list1: List[Document], list2: List[Document], k: int = 60) -> List[Document]:
        """Reciprocal Rank Fusion to merge two ranked lists."""
        scores = {}
        for rank, doc in enumerate(list1):
            key = doc.page_content[:50]
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
            scores[key + "_doc"] = doc
        for rank, doc in enumerate(list2):
            key = doc.page_content[:50]
            scores[key] = scores.get(key, 0) + 1 / (k + rank + 1)
            scores[key + "_doc"] = doc
        
        # Sort by RRF score
        doc_keys = [k for k in scores if not k.endswith("_doc")]
        sorted_keys = sorted(doc_keys, key=lambda x: scores[x], reverse=True)
        return [scores[k + "_doc"] for k in sorted_keys[:self.k]]

retriever = HybridRetriever(vectorstore, chunks, k=3)

# Test
results = retriever.retrieve("How does RAG work?")
print(f"🔍 Retrieved {len(results)} chunks for 'How does RAG work?'\n")
for i, doc in enumerate(results):
    print(f"  [{i+1}] {doc.page_content[:80]}...")
    print(f"      Source: {doc.metadata.get('source', 'unknown')}\n")

## 🧪 Part 3: RAG Chain with Citations

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Structured output with citations
class CitedAnswer(BaseModel):
    answer: str = Field(description="The answer to the question")
    sources: List[str] = Field(description="Source documents used")
    confidence: str = Field(description="high, medium, or low")

RAG_PROMPT = ChatPromptTemplate.from_template("""
Answer the question using ONLY the context below.
Include which sources you used. If unsure, say so.

Context:
{context}

Question: {question}
""")

structured_llm = llm.with_structured_output(CitedAnswer)

def ask_with_citations(question: str) -> CitedAnswer:
    # Retrieve
    docs = retriever.retrieve(question)
    context = "\n\n".join(
        f"[Source: {d.metadata.get('source', 'unknown')}]\n{d.page_content}" 
        for d in docs
    )
    
    # Generate
    prompt = RAG_PROMPT.invoke({"context": context, "question": question})
    return structured_llm.invoke(prompt)

# Test
result = ask_with_citations("What are advanced RAG techniques?")
print(f"💬 Answer: {result.answer}\n")
print(f"📚 Sources: {result.sources}")
print(f"🎯 Confidence: {result.confidence}")

## 🧪 Part 4: Conversation Memory

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

class RAGAssistant:
    """RAG chatbot with conversation memory."""
    
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history: List = []
    
    def chat(self, question: str) -> str:
        # Retrieve context
        docs = self.retriever.retrieve(question)
        context = "\n".join(d.page_content for d in docs)
        
        # Build messages with history
        messages = [
            {"role": "system", "content": f"""You are a helpful assistant. Answer using the context below.
If the answer isn't in the context, say so.

Context:
{context}"""}
        ]
        
        # Add conversation history (last 6 messages)
        for msg in self.history[-6:]:
            messages.append(msg)
        
        messages.append({"role": "user", "content": question})
        
        # Generate
        response = self.llm.invoke(messages)
        answer = response.content
        
        # Save to history
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        return answer
    
    def reset(self):
        self.history = []

# Test conversation
assistant = RAGAssistant(retriever, llm)

print("💬 Conversation Test:\n")
q1 = "What is RAG?"
a1 = assistant.chat(q1)
print(f"User: {q1}")
print(f"Bot: {a1}\n")

q2 = "What advanced techniques can improve it?"
a2 = assistant.chat(q2)
print(f"User: {q2}")
print(f"Bot: {a2}\n")

q3 = "Which vector database should I use?"
a3 = assistant.chat(q3)
print(f"User: {q3}")
print(f"Bot: {a3}")

## 🧪 Part 5: Answer Quality Evaluation

In [ ]:
EVAL_PROMPT = ChatPromptTemplate.from_template("""
Evaluate this RAG response on two criteria. Score each 1-5.

Question: {question}
Context provided: {context}
Answer given: {answer}

Criteria:
1. FAITHFULNESS: Is the answer grounded in the context? (1=hallucinated, 5=fully grounded)
2. RELEVANCE: Does it answer the question? (1=off-topic, 5=perfectly answers)

Respond as JSON: {{"faithfulness": N, "relevance": N, "explanation": "..."}}
""")

def evaluate_answer(question: str, context: str, answer: str) -> dict:
    prompt = EVAL_PROMPT.invoke({
        "question": question,
        "context": context,
        "answer": answer
    })
    result = llm.invoke(prompt)
    import json
    try:
        return json.loads(result.content)
    except json.JSONDecodeError:
        return {"raw": result.content}

# Evaluate our RAG answer
docs = retriever.retrieve("What is RAG?")
context = "\n".join(d.page_content for d in docs)
answer = assistant.chat("What is RAG?")

eval_result = evaluate_answer("What is RAG?", context, answer)
print("📊 Evaluation Result:")
for key, value in eval_result.items():
    print(f"  {key}: {value}")

## ✅ Capstone 1 Complete!

```
┌───────────────────────────────────────────────────────────────────┐
│  📌 Built: Document ingestion → Hybrid retrieval → Cited RAG    │
├───────────────────────────────────────────────────────────────────┤
│  📌 Added: Conversation memory for follow-up questions           │
├───────────────────────────────────────────────────────────────────┤
│  📌 Evaluated: LLM-as-Judge for faithfulness & relevance        │
├───────────────────────────────────────────────────────────────────┤
│  📌 Production-ready: Hybrid search, structured output, eval    │
└───────────────────────────────────────────────────────────────────┘
```

---

### 🏆 RAG Knowledge Assistant — Done!
From ingestion to evaluation — a complete RAG system. 📚